# Analyse exploratoire — Churn Telco

**Objectif métier :** repérer les clients à risque de départ pour lancer des actions de rétention avec un horizon d’action en amont (signaux : contrat court, charges élevées, faible ancienneté, etc.).  
**Données :** jeu IBM Telco (`WA_Fn-UseC_-Telco-Customer-Churn`).  
**Note :** il n’y a pas de variable « âge » ; on utilise `SeniorCitizen` (0/1), `tenure` et les charges.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="husl")
plt.rcParams["figure.figsize"] = (8, 4)

DATA_PATH = "../WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(DATA_PATH)
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df["TotalCharges"] = df["TotalCharges"].fillna(0.0)
df["Churn_bin"] = (df["Churn"] == "Yes").astype(int)
df.head()

## 1. Distributions (proxy âge, ancienneté, charges)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 8))
sns.histplot(df["SeniorCitizen"], discrete=True, ax=axes[0, 0])
axes[0, 0].set_title("SeniorCitizen (proxy statut senior)")
sns.histplot(df["tenure"], bins=30, kde=True, ax=axes[0, 1])
axes[0, 1].set_title("Ancienneté (mois)")
sns.histplot(df["MonthlyCharges"], bins=35, kde=True, ax=axes[1, 0])
axes[1, 0].set_title("Charges mensuelles")
sns.histplot(df["TotalCharges"], bins=40, kde=True, ax=axes[1, 1])
axes[1, 1].set_title("Charges cumulées")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, col, title in zip(
    axes,
    ["tenure", "MonthlyCharges", "TotalCharges"],
    ["Tenure", "MonthlyCharges", "TotalCharges"],
):
    sns.kdeplot(data=df, x=col, hue="Churn", common_norm=False, ax=ax, fill=True, alpha=0.35)
    ax.set_title(f"{title} vs Churn")
plt.tight_layout()
plt.show()

## 2. Corrélations numériques vs churn

In [ ]:
num_cols = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges", "Churn_bin"]
corr = df[num_cols].corr()
plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1)
plt.title("Corrélations (Pearson) — variables numériques")
plt.show()
print("Corrélation avec churn :")
print(corr["Churn_bin"].drop("Churn_bin").sort_values(key=abs, ascending=False))

## 3. Variables catégorielles vs taux de churn

In [ ]:
cat_cols = [
    "Contract", "InternetService", "PaymentMethod",
    "OnlineSecurity", "TechSupport", "PaperlessBilling",
]
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.ravel()
for ax, c in zip(axes, cat_cols):
    rate = df.groupby(c)["Churn_bin"].mean().sort_values(ascending=False)
    rate.plot(kind="bar", ax=ax, color="steelblue")
    ax.set_title(f"Taux churn par {c}")
    ax.set_ylabel("P(Churn)")
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 4. Insights business actionnables

- **Contrat month-to-month** : souvent associé à un churn plus élevé → proposer migration vers offres 12/24 mois avec avantage mesurable.
- **Fibre sans options sécurité / support** : segments à risque → pack « tranquillité » (support + sécurité) ou contact proactif.
- **Paiement par chèque électronique** : friction perçue → inciter aux prélèvements avec réduction ou bonus data.
- **Faible tenure + charges mensuelles élevées** : clients récents sous tension prix → offre de fidélisation ciblée les 3–6 premiers mois.
- **Senior citizen** : à croiser avec canaux préférés pour éviter les faux négatifs sur populations moins digitales.

*Ces axes peuvent alimenter des campagnes de rétention et la priorisation des appels sortants sur les scores modèle (voir pipeline ML + dashboard).*